# Battery Report

Analysis of one or more CarbonShift solver battery runs. Pick which run
folder(s) under `results/` to analyse in the cell below (auto-detected from
`battery_results_*.csv` files); no manual path editing needed.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

plt.rcParams["figure.dpi"] = 96
plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.4f}".format)

STRATEGY_STYLE: dict = {
    "dp": {"color": "tab:orange", "marker": "o", "linestyle": "-"},
    "offline_singleton_best": {"color": "tab:green", "marker": "D", "linestyle": "none"},
    "offline_singleton_best_accurate": {"color": "tab:olive", "marker": "s", "linestyle": "none"},
    "offline_bandit": {"color": "tab:purple", "marker": "^", "linestyle": "none"},
    "offline_ant_colony": {"color": "tab:cyan", "marker": "*", "linestyle": "none"},
}
ONLINE_STRATEGY_STYLE: dict = {
    "online_singleton_best": {"color": "tab:green", "linestyle": "--", "marker": "D"},
    "online_bandit": {"color": "tab:purple", "linestyle": "--", "marker": "^"},
    "online_ant_colony": {"color": "tab:cyan", "linestyle": "--", "marker": "*"},
}
_STRATEGY_FALLBACK_COLORS = ["tab:brown", "tab:pink", "tab:olive"]

METRICS: list[dict] = [
    {"col": "solver_time_ms_avg", "label": "Solver time (ms avg)"},
    {"col": "carbon_cost", "label": "Total carbon cost"},
    {"col": "final_global_error", "label": "Final global error"},
    {"col": "carbon_saving", "label": "Carbon saving vs baseline (%)"},
    {"col": "reqs_fallback_assigned", "label": "Requests assigned with fallback"},
    {"col": "peak_concurrent_workers", "label": "Peak concurrent workers"},
    {"col": "avg_concurrent_workers", "label": "Avg concurrent workers"},
]


In [ ]:
from pathlib import Path
import sys
import re

candidate_root = Path.cwd()
if (candidate_root / "tests" / "battery" / "results").exists():
    BATTERY_DIR = candidate_root / "tests" / "battery"
elif (candidate_root / "results").exists():
    BATTERY_DIR = candidate_root
elif "__file__" in globals():
    BATTERY_DIR = Path(__file__).resolve().parent
else:
    raise FileNotFoundError("Run this notebook from the repository root or tests/battery")

RESULTS_ROOT = BATTERY_DIR / "results"
sys.path.insert(0, str(BATTERY_DIR))
from nb_utils import require_output_schema


def _find_run_dirs(root: Path) -> list[Path]:
    """Find battery-run folders containing result CSVs, newest first."""
    dirs = {p.parent for p in root.glob("**/battery_results_*.csv")}
    dirs = [directory for directory in dirs if "archive" not in directory.parts]
    return sorted(
        dirs,
        key=lambda directory: max(
            (file.stat().st_mtime for file in directory.glob("battery_results_*.csv")),
            default=0,
        ),
        reverse=True,
    )


_available_runs = _find_run_dirs(RESULTS_ROOT)
print(f"Found {len(_available_runs)} run folder(s) under {RESULTS_ROOT}:")
for i, directory in enumerate(_available_runs):
    relative = directory.relative_to(RESULTS_ROOT) if directory != RESULTS_ROOT else Path(".")
    print(f"  [{i}] {relative}")

SELECTED_RUNS: list[int] = [0]
RESULT_CSVS: list[Path] = [
    file
    for i in SELECTED_RUNS
    for file in sorted(_available_runs[i].glob("battery_results_*.csv"))
]
print(f"\nAnalysing: {[str(path.relative_to(RESULTS_ROOT)) for path in RESULT_CSVS]}")


In [ ]:
frames = []
required_columns = {
    "scenario_id",
    "batch_size",
    "strategy",
    "solver_time_ms_avg",
    "carbon_cost",
    "final_global_error",
    "reqs_in",
    "reqs_out",
    "reqs_fallback_assigned",
    "baseline_carbon_cost",
    "carbon_saving",
    "peak_concurrent_workers",
    "avg_concurrent_workers",
}
for path in RESULT_CSVS:
    if not path.is_absolute():
        path = (BATTERY_DIR / path).resolve()
    if not path.exists():
        raise FileNotFoundError(f"Results CSV not found: {path}")
    require_output_schema(path, schema_name="carbonshift.battery-run")
    frame = pd.read_csv(path)
    missing = required_columns - set(frame.columns)
    if missing:
        raise ValueError(f"Schema-v2 battery results are missing columns: {sorted(missing)}")
    battery_id = path.stem.removeprefix("battery_results").lstrip("_") or "unnamed"
    frame.insert(0, "battery_id", battery_id)
    frame["backend"] = "rust"
    frames.append(frame)

if not frames:
    raise ValueError("Select at least one schema-v2 battery result CSV")
df = pd.concat(frames, ignore_index=True)
df["batch_size"] = pd.to_numeric(df["batch_size"], errors="raise").astype(int)
df = df.drop_duplicates(
    subset=["battery_id", "scenario_id", "strategy", "batch_size"], keep="first"
)

ONLINE_STRATEGY_NAMES = sorted(
    strategy for strategy in df["strategy"].unique() if strategy.startswith("online_")
)
STRATEGY_NAMES = sorted(
    strategy for strategy in df["strategy"].unique() if strategy.startswith("offline_")
)
df_online = df[df["strategy"].isin(ONLINE_STRATEGY_NAMES)].copy()
df_strat = df[df["strategy"].isin(STRATEGY_NAMES)].copy()
df_modes = df[~df["strategy"].str.startswith(("online_", "offline_"))].copy()
scenarios = sorted(df["scenario_id"].unique())
_online_batch_sizes = set(df_online["batch_size"].unique())
batch_sizes = sorted(set(df_modes["batch_size"].unique()) | _online_batch_sizes)
combos = sorted(
    df_modes[["backend", "strategy"]].drop_duplicates().itertuples(index=False, name=None)
)


def _xaxis_info():
    """Build categorical x-axis covering batch sizes and offline strategies."""
    labels = [f"N={n}" for n in batch_sizes] + STRATEGY_NAMES
    batch_to_x = {n: i for i, n in enumerate(batch_sizes)}
    strategy_to_x = {name: len(batch_sizes) + i for i, name in enumerate(STRATEGY_NAMES)}
    return labels, batch_to_x, strategy_to_x, len(batch_sizes)


x_labels, bs_to_x, strat_to_x, n_bs = _xaxis_info()
print(
    f"Loaded {len(df)} rows · batteries={sorted(df['battery_id'].unique())} · "
    f"scenarios={len(scenarios)} · N={batch_sizes} · "
    f"DP strategies={sorted(df_modes['strategy'].unique())} · "
    f"offline={STRATEGY_NAMES} · online={ONLINE_STRATEGY_NAMES}"
)
df


## Execution timings

Wall-clock time per scenario and per batch-size, from the matching `battery_timings_*.csv` / `per_n_timings_*.csv` files.

In [ ]:
timing_frames = []
per_n_frames = []
for p in RESULT_CSVS:
    p = Path(p) if not isinstance(p, Path) else p
    if not p.is_absolute():
        p = (_nb_dir / p).resolve()
    stem = p.stem
    bid = stem.removeprefix("battery_results").lstrip("_") or "unnamed"
    timing_path = p.parent / f"battery_timings{('_' + bid) if bid != 'unnamed' else ''}.csv"
    per_n_path  = p.parent / f"per_n_timings{('_' + bid) if bid != 'unnamed' else ''}.csv"
    if not timing_path.exists():
        print(f"WARNING: timing file not found: {timing_path}")
        continue
    _t = pd.read_csv(timing_path)
    _t.insert(0, "battery_id", bid)
    timing_frames.append(_t)
    if per_n_path.exists():
        _pn = pd.read_csv(per_n_path)
        _pn.insert(0, "battery_id", bid)
        per_n_frames.append(_pn)

if not timing_frames:
    print("No timing files found – run the battery first to generate timings.")
else:
    tdf = pd.concat(timing_frames, ignore_index=True)
    scenario_times = tdf[tdf["scenario_id"] != "__battery_total__"].copy()
    battery_totals = tdf[tdf["scenario_id"] == "__battery_total__"].copy()
    battery_ids = scenario_times["battery_id"].unique().tolist()
    n_batteries = len(battery_ids)

    # ── Chart 1: total wall-clock per scenario ────────────────────────────────
    fig, axes = plt.subplots(1, n_batteries, figsize=(max(7, 5 * n_batteries), 4.5), squeeze=False)
    for ax, bid in zip(axes[0], battery_ids):
        sub = scenario_times[scenario_times["battery_id"] == bid].sort_values("scenario_id")
        total_row = battery_totals[battery_totals["battery_id"] == bid]
        total_s = float(total_row["elapsed_seconds"].iloc[0]) if not total_row.empty else sub["elapsed_seconds"].sum()
        bars = ax.barh(sub["scenario_id"], sub["elapsed_seconds"], color="steelblue", edgecolor="white")
        for bar, val in zip(bars, sub["elapsed_seconds"]):
            ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
                    f"{val:.1f}s", va="center", fontsize=9)
        ax.set_xlabel("Wall-clock time (seconds)")
        ax.set_title(f"Battery: {bid}\nTotal: {total_s:.1f}s", fontsize=11, fontweight="bold")
        ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.show()
    print("\nPer-battery totals:")
    for bid in battery_ids:
        total_row = battery_totals[battery_totals["battery_id"] == bid]
        if not total_row.empty:
            print(f"  {bid}: {float(total_row['elapsed_seconds'].iloc[0]):.1f}s")

# ── Chart 2: per-N/strategy stacked breakdown ─────────────────────────────────
if per_n_frames:
    pndf = pd.concat(per_n_frames, ignore_index=True)
    battery_ids_pn = pndf["battery_id"].unique().tolist()

    # Build a color map: N values get a sequential palette, strategies get distinct colors
    all_modes = sorted(pndf["strategy"].unique().tolist())
    all_ns    = sorted(pndf["batch_size"].unique().tolist())

    # Assign colors: use tab20 for many items
    cmap_n = plt.get_cmap("tab20")
    mode_color = {}
    for i, m in enumerate(all_modes):
        mode_color[m] = cmap_n(i / max(len(all_modes), 1))

    fig, axes = plt.subplots(1, len(battery_ids_pn),
                              figsize=(max(8, 6 * len(battery_ids_pn)), max(3, len(pndf["scenario_id"].unique()) * 0.7 + 2)),
                              squeeze=False)
    for ax, bid in zip(axes[0], battery_ids_pn):
        sub = pndf[pndf["battery_id"] == bid].copy()
        scenarios = sorted(sub["scenario_id"].unique().tolist())
        lefts = {s: 0.0 for s in scenarios}

        # Sort modes so N values come first (numerically), then strategies
        dp_modes = sorted([m for m in sub["strategy"].unique() if not m.startswith(("online_", "offline_"))])
        strat_modes = sorted([m for m in sub["strategy"].unique() if m not in dp_modes])
        ordered_modes = dp_modes + strat_modes

        for mode in ordered_modes:
            vals = []
            for s in scenarios:
                row = sub[(sub["scenario_id"] == s) & (sub["strategy"] == mode)]
                if row.empty:
                    vals.append(0.0)
                else:
                    # Sum all batch sizes for this mode (stacked within mode)
                    vals.append(float(row["elapsed_seconds"].sum()))

            color = mode_color.get(mode, "gray")
            bars = ax.barh(scenarios, vals,
                           left=[lefts[s] for s in scenarios],
                           color=color, label=mode, edgecolor="white", linewidth=0.5)
            for s, val in zip(scenarios, vals):
                if val > 1.0:
                    x = lefts[s] + val / 2
                    ax.text(x, scenarios.index(s), f"{val:.0f}s",
                            ha="center", va="center", fontsize=7.5, color="white", fontweight="bold")
            for s, val in zip(scenarios, vals):
                lefts[s] += val

        ax.set_xlabel("Wall-clock time (seconds)")
        ax.set_title(f"Per-mode timing breakdown\n(battery: {bid})", fontsize=11, fontweight="bold")
        ax.legend(loc="lower right", fontsize=8)
        ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.show()

    # ── Chart 3: per-N breakdown per mode (N values stacked) ─────────────────
    for bid in battery_ids_pn:
        sub = pndf[pndf["battery_id"] == bid].copy()
        dp_sub = sub[~sub["strategy"].str.startswith(("online_", "offline_"))]
        if dp_sub.empty:
            continue
        modes_present = sorted(dp_sub["strategy"].unique().tolist())
        scenarios = sorted(dp_sub["scenario_id"].unique().tolist())

        fig, axes2 = plt.subplots(1, len(modes_present),
                                   figsize=(max(7, 5 * len(modes_present)), max(3, len(scenarios) * 0.8 + 2)),
                                   squeeze=False)
        for ax, mode in zip(axes2[0], modes_present):
            msub = dp_sub[dp_sub["strategy"] == mode].copy()
            ns_present = sorted(msub["batch_size"].unique().tolist())
            n_colors = plt.get_cmap("viridis")(
                [i / max(len(ns_present) - 1, 1) for i in range(len(ns_present))]
            )
            lefts2 = {s: 0.0 for s in scenarios}
            for ni, (n, col) in enumerate(zip(ns_present, n_colors)):
                vals2 = []
                for s in scenarios:
                    row = msub[(msub["scenario_id"] == s) & (msub["batch_size"] == n)]
                    vals2.append(float(row["elapsed_seconds"].iloc[0]) if not row.empty else 0.0)
                ax.barh(scenarios, vals2, left=[lefts2[s] for s in scenarios],
                        color=col, label=f"N={n}", edgecolor="white", linewidth=0.4)
                for s, val in zip(scenarios, vals2):
                    if val > 2.0:
                        x = lefts2[s] + val / 2
                        ax.text(x, scenarios.index(s), f"{val:.0f}s",
                                ha="center", va="center", fontsize=7, color="white", fontweight="bold")
                for s, val in zip(scenarios, vals2):
                    lefts2[s] += val
            ax.set_xlabel("Wall-clock time (seconds)")
            ax.set_title(f"{mode}\n(battery: {bid})", fontsize=10, fontweight="bold")
            ax.legend(loc="lower right", fontsize=7)
            ax.grid(axis="x", alpha=0.3)
        plt.tight_layout()
        plt.show()
else:
    print("No per-N timing files found – run the battery again to generate per_n_timings_*.csv.")

# ── Strategy execution timing ─────────────────────────────────────────────────
if not df_strat.empty and "solver_time_ms_avg" in df_strat.columns:
    print("\n### Strategy execution times (ms)")
    timing_strat = (
        df_strat
        .groupby("strategy")["solver_time_ms_avg"]
        .agg(min="min", mean="mean", max="max")
        .reset_index()
        .rename(columns={"strategy": "strategy"})
    )

    fig, ax = plt.subplots(figsize=(max(4, len(STRATEGY_NAMES) * 1.8 + 1), 4))
    for i, strat in enumerate(STRATEGY_NAMES):
        row = timing_strat[timing_strat["strategy"] == strat]
        if row.empty:
            continue
        mean_val = float(row["mean"])
        min_val  = float(row["min"])
        max_val  = float(row["max"])
        color = STRATEGY_STYLE.get(strat, {}).get("color", _STRATEGY_FALLBACK_COLORS[i % len(_STRATEGY_FALLBACK_COLORS)])
        ax.bar(i, mean_val, color=color, alpha=0.8, label=strat)
        ax.plot([i, i], [min_val, max_val], color="black", linewidth=2.5, zorder=5)
        ax.scatter([i], [mean_val], color="black", s=30, zorder=6)

    # Reference lines: avg solver time for each mode
    if not df_modes.empty:
        mode_colors = {"greedy_fallback": "tab:orange", "relaxed_retry": "tab:red"}
        for mode, grp in df_modes.groupby("strategy"):
            avg = grp["solver_time_ms_avg"].mean()
            ax.axhline(avg, linestyle="--", linewidth=1.4, alpha=0.8,
                       color=mode_colors.get(mode, "gray"),
                       label=f"{mode} avg ({avg:.1f}ms)")

    ax.set_xticks(range(len(STRATEGY_NAMES)))
    ax.set_xticklabels(STRATEGY_NAMES, rotation=20, ha="right")
    ax.set_ylabel("Solver time (ms avg)")
    ax.set_title("Strategy execution time (bar=mean, whisker=min/max)", fontsize=11, fontweight="bold")
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()
    display(timing_strat.round(3))
else:
    print("No strategy data loaded – run battery with additional_strategies to see timings.")


In [ ]:
def _series_label(backend: str, strategy: str) -> str:
    return f"{backend}/{strategy}"


def _series_style(backend: str, strategy: str) -> dict:
    style = STRATEGY_STYLE.get(strategy, {})
    return {"color": style.get("color", "gray"), **style}


def _strategy_style(name: str) -> dict:
    if name in STRATEGY_STYLE:
        return STRATEGY_STYLE[name]
    i = STRATEGY_NAMES.index(name) % len(_STRATEGY_FALLBACK_COLORS)
    return {"color": _STRATEGY_FALLBACK_COLORS[i], "marker": "X"}


def _online_strategy_style(name: str) -> dict:
    if name in ONLINE_STRATEGY_STYLE:
        return ONLINE_STRATEGY_STYLE[name]
    i = ONLINE_STRATEGY_NAMES.index(name) % len(_STRATEGY_FALLBACK_COLORS)
    return {"color": _STRATEGY_FALLBACK_COLORS[i], "linestyle": "--", "marker": "^"}


def _apply_xaxis(ax, rotate=30):
    ax.set_xticks(range(len(x_labels)))
    ax.set_xticklabels(x_labels, rotation=rotate, ha="right", fontsize=8)
    if STRATEGY_NAMES and n_bs > 0:
        ax.axvline(x=n_bs - 0.5, linestyle=":", linewidth=0.9, alpha=0.6, color="gray")
        ax.text(n_bs - 0.55, ax.get_ylim()[1], "\u2190 DP / Online", ha="right", va="top", fontsize=6, color="gray", style="italic")
        ax.text(n_bs - 0.45, ax.get_ylim()[1], "Offline \u2192", ha="left", va="top", fontsize=6, color="gray", style="italic")


## Metric comparison across scenarios

One figure per metric; each panel is a `(backend, strategy)` combo (or online strategy), with one line per scenario and offline-strategy scatter points on the right.

In [ ]:
# Build panels: one per DP combo, then one per online strategy
_panels_8 = list(combos) + [("rust", ostrat) for ostrat in ONLINE_STRATEGY_NAMES]

_n_panels = max(len(_panels_8), 1)
_fig_w = max(6, 4.5 * _n_panels)

colors_cycle = plt.rcParams["axes.prop_cycle"].by_key()["color"]
sid_color = {sid: colors_cycle[i % len(colors_cycle)] for i, sid in enumerate(scenarios)}

for m in METRICS:
    fig, axes = plt.subplots(1, _n_panels, figsize=(_fig_w, 4.5), sharey=True)
    if _n_panels == 1:
        axes = [axes]
    fig.suptitle(m["label"], fontsize=13, fontweight="bold")

    for ax, (pbackend, pmode) in zip(axes, _panels_8):
        is_online = pmode.startswith("online_")
        ax.set_title(pmode if is_online else _series_label(pbackend, pmode), fontsize=10)
        if ax is axes[0]:
            ax.set_ylabel(m["label"])

        for sid in scenarios:
            if is_online:
                sub = df_online[
                    (df_online["scenario_id"] == sid) &
                    (df_online["strategy"] == pmode)
                ].sort_values("batch_size")
            else:
                sub = df_modes[
                    (df_modes["scenario_id"] == sid) &
                    (df_modes["backend"] == pbackend) &
                    (df_modes["strategy"] == pmode)
                ].sort_values("batch_size")
            if sub.empty or m["col"] not in sub.columns:
                continue
            xs = [bs_to_x[bs] for bs in sub["batch_size"] if bs in bs_to_x]
            ys = [float(sub[sub["batch_size"] == bs][m["col"]].values[0])
                  for bs in sub["batch_size"] if bs in bs_to_x]
            if not xs:
                continue
            ax.plot(xs, ys, marker="o", linewidth=1.6, color=sid_color[sid], label=sid)
            # Dashed extension to offline strategy scatter points
            if not df_strat.empty and STRATEGY_NAMES:
                ext_xs, ext_ys = [xs[-1]], [ys[-1]]
                for strat in STRATEGY_NAMES:
                    sub_s = df_strat[
                        (df_strat["scenario_id"] == sid) &
                        (df_strat["strategy"] == strat)
                    ]
                    if sub_s.empty or m["col"] not in sub_s.columns:
                        continue
                    ext_xs.append(strat_to_x[strat])
                    ext_ys.append(float(sub_s[m["col"]].values[0]))
                if len(ext_xs) > 1:
                    ax.plot(ext_xs, ext_ys, linestyle="--", linewidth=1.2,
                            alpha=0.7, color=sid_color[sid])

        # Offline strategy scatter (one marker per strategy, colour by scenario)
        for strat in STRATEGY_NAMES:
            sty = _strategy_style(strat)
            for sid in scenarios:
                sub_s = df_strat[
                    (df_strat["scenario_id"] == sid) &
                    (df_strat["strategy"] == strat)
                ]
                if sub_s.empty or m["col"] not in sub_s.columns:
                    continue
                ax.scatter([strat_to_x[strat]], [float(sub_s[m["col"]].values[0])],
                           s=70, zorder=5, color=sid_color[sid], marker=sty["marker"])
            ax.scatter([], [], s=70, marker=sty["marker"], color=sty["color"], label=f"\u25b8{strat}")

        _apply_xaxis(ax)
        ax.legend(fontsize=7, loc="best")

    plt.tight_layout()
    plt.show()


## Aggregate statistics

Min / mean / max of each metric across all loaded scenarios, grouped by `(backend, strategy, batch_size)`.

In [ ]:
metric_cols = [m["col"] for m in METRICS]
agg_modes = (
    df_modes.groupby(["backend", "strategy", "batch_size"])[metric_cols]
    .agg(["min", "mean", "max"])
    .round(4)
)
print("=== Modes: min / mean / max per (backend, mode, batch_size) ===")
display(agg_modes)

if not df_strat.empty:
    agg_strat = (
        df_strat.groupby(["strategy"])[metric_cols]
        .agg(["min", "mean", "max"])
        .round(4)
    )
    print("\n=== Offline strategies: min / mean / max across all scenarios ===")
    display(agg_strat)

if not df_online.empty:
    agg_online = (
        df_online.groupby(["strategy", "batch_size"])[metric_cols]
        .agg(["min", "mean", "max"])
        .round(4)
    )
    print("\n=== Online Strategies: min / mean / max per (strategy, N) ===")
    display(agg_online)


In [ ]:
combos_sorted = sorted(combos)
fig_rows = len(METRICS)
fig_cols = max(len(combos_sorted), 1)

fig, axes = plt.subplots(
    fig_rows, fig_cols,
    figsize=(4.8 * fig_cols, 4.0 * fig_rows),
    squeeze=False,
    sharey="row",
)
fig.suptitle("Aggregate statistics across all scenarios (shaded = min–max range)\n",
             fontsize=13, fontweight="bold")

for row, m in enumerate(METRICS):
    for col, (backend, mode) in enumerate(combos_sorted):
        ax = axes[row][col]
        sub = df_modes[(df_modes["backend"] == backend) & (df_modes["strategy"] == mode)]
        if sub.empty:
            ax.set_visible(False)
            continue
        grp = (
            sub.groupby("batch_size")[m["col"]]
            .agg(["min", "mean", "max"])
            .reset_index()
            .sort_values("batch_size")
        )
        xs = [bs_to_x[bs] for bs in grp["batch_size"]]
        color = BACKEND_COLOR.get(backend, {}).get(mode, "gray")
        ax.plot(xs, grp["mean"], marker="o", linewidth=2.0, color=color, label="mean")
        ax.fill_between(xs, grp["min"], grp["max"], alpha=0.2, color=color, label="min–max")
        ax.plot(xs, grp["min"], linestyle=":", linewidth=1.0, color=color, alpha=0.7)
        ax.plot(xs, grp["max"], linestyle=":", linewidth=1.0, color=color, alpha=0.7)

        # Online strategy lines: mean ± fill across scenarios per N
        for ostrat in ONLINE_STRATEGY_NAMES:
            if df_online.empty:
                continue
            sub_o = df_online[df_online["strategy"] == ostrat]
            if sub_o.empty or m["col"] not in sub_o.columns:
                continue
            grp_o = (
                sub_o.groupby("batch_size")[m["col"]]
                .agg(["min", "mean", "max"])
                .reset_index().sort_values("batch_size")
            )
            xs_o = [bs_to_x[bs] for bs in grp_o["batch_size"] if bs in bs_to_x]
            sty_o = _online_strategy_style(ostrat)
            means_o = [float(grp_o[grp_o["batch_size"] == bs]["mean"].values[0])
                       for bs in grp_o["batch_size"] if bs in bs_to_x]
            mins_o  = [float(grp_o[grp_o["batch_size"] == bs]["min"].values[0])
                       for bs in grp_o["batch_size"] if bs in bs_to_x]
            maxs_o  = [float(grp_o[grp_o["batch_size"] == bs]["max"].values[0])
                       for bs in grp_o["batch_size"] if bs in bs_to_x]
            if xs_o:
                ax.plot(xs_o, means_o, linewidth=1.8, label=ostrat, **sty_o)
                ax.fill_between(xs_o, mins_o, maxs_o, alpha=0.15, color=sty_o["color"])

        # Strategy points: mean ± range across scenarios
        for strat in STRATEGY_NAMES:
            if df_strat.empty:
                continue
            vals = df_strat[df_strat["strategy"] == strat][m["col"]].dropna()
            if vals.empty:
                continue
            x_pos = strat_to_x[strat]
            sty = _strategy_style(strat)
            sc = ax.scatter([x_pos], [vals.mean()], s=70, zorder=5,
                            color=sty["color"], marker=sty["marker"], label=strat)
            ax.plot([x_pos, x_pos], [vals.min(), vals.max()],
                    color=sty["color"], linewidth=2.5, alpha=0.6)

        if STRATEGY_NAMES and n_bs > 0:
            ax.axvline(x=n_bs - 0.5, linestyle=":", linewidth=0.8, alpha=0.5, color="gray")
        ax.set_xticks(range(len(x_labels)))
        ax.set_xticklabels(x_labels, rotation=30, ha="right", fontsize=7)
        ax.set_xlabel("N / strategy")
        if row == 0:
            ax.set_title(_series_label(backend, mode), fontsize=10)
        if col == 0:
            ax.set_ylabel(m["label"], fontsize=9)
        ax.legend(fontsize=7, loc="best")

plt.tight_layout()
plt.show()


In [ ]:
metric_cols = [m["col"] for m in METRICS]

overall_modes = (
    df_modes.groupby(["backend", "strategy"])[metric_cols]
    .agg(["min", "mean", "max"])
    .round(4)
)
print("Overall min / mean / max across ALL scenarios and ALL N values (modes):")
display(overall_modes)

if not df_strat.empty:
    overall_strat = (
        df_strat.groupby("strategy")[metric_cols]
        .agg(["min", "mean", "max"])
        .round(4)
    )
    print("\nOverall min / mean / max across ALL scenarios (strategies):")
    display(overall_strat)
